In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()
tools

[StructuredTool(name='search-flight', description='# Search for flights\n\nSearches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.\n\n## Result shape\n\nReturns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:\n- `price` (number) and `priceFormatted` (e.g. "123 EUR")\n- `totalDurationSeconds`\n- `bookingUrl` — the link to book the flight\n- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)\n- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)\n- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (loc

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite"),
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from Ahmedabad to Delhi on 10th august 2026")], "config": config},
    config
    )

In [9]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='23094c65-0dd3-44f4-b211-9183199e2574'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search-flight', 'arguments': '{"flyFrom": "SFO", "departureDate": "31/03/2025", "flyTo": "TYO"}'}, '__gemini_function_call_thought_signatures__': {'1a5d87e4-c412-4cc9-9e74-7e3866ce67c1': 'EjQKMgERTTIPz4JqzDXAnybpL/wUTHRzx5ImCo6Sai1ExEugw+cfxNgZEuHK3AswcAl2Z6VB'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fad71-d95e-7f43-bd54-b8e2efaf96ab-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'SFO', 'departureDate': '31/03/2025', 'flyTo': 'TYO'}, 'id': '1a5d87e4-c412-4cc9-9e74-7e3866ce67c1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2649, 'output_tokens': 42, 'tota

In [10]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': "Here are the available direct flight options from Ahmedabad (AMD) to Delhi (DEL) on August 10th, 2026.\n\n| Route | Times & Duration | Price | Booking |\n| :--- | :--- | :--- | :--- |\n| **AMD → DEL** | 10 Aug 19:25 → 21:05 (1h 40m) | 55 EUR | [Book Now](https://kiwi.com/u/k2xnuc) |\n| **AMD → DEL** | 10 Aug 21:50 → 23:30 (1h 40m) | 55 EUR | [Book Now](https://kiwi.com/u/vgttvdu) |\n| **AMD → DEL** | 10 Aug 05:00 → 06:45 (1h 45m) | 54 EUR | [Book Now](https://kiwi.com/u/dh9jc5) |\n| **AMD → DEL** | 10 Aug 07:30 → 09:15 (1h 45m) | 54 EUR | [Book Now](https://kiwi.com/u/r5qj5c) |\n| **AMD → DEL** | 10 Aug 09:50 → 11:35 (1h 45m) | 54 EUR | [Book Now](https://kiwi.com/u/2fkz5j) |\n| **AMD → DEL** | 10 Aug 14:35 → 16:20 (1h 45m) | 54 EUR | [Book Now](https://kiwi.com/u/8n7qu4) |\n\n### Summary\n*   **Cheapest Options:** There are several direct flights priced at **54 EUR**.\n*   **Shortest Options:** The flights departing at 19:25 and 21:50 are the shortest at **1

In [ ]:
from rich import print as rprint
rprint(response["messages"][-1].content)


[
    {
        'type': 'text',
        'text': "Here are the available direct flight options from Ahmedabad (AMD) to Delhi (DEL) on August 10th, 
2026.\n\n| Route | Times & Duration | Price | Booking |\n| :--- | :--- | :--- | :--- |\n| **AMD → DEL** | 10 Aug 
19:25 → 21:05 (1h 40m) | 55 EUR | [Book Now](https://kiwi.com/u/k2xnuc) |\n| **AMD → DEL** | 10 Aug 21:50 → 23:30 
(1h 40m) | 55 EUR | [Book Now](https://kiwi.com/u/vgttvdu) |\n| **AMD → DEL** | 10 Aug 05:00 → 06:45 (1h 45m) | 54 
EUR | [Book Now](https://kiwi.com/u/dh9jc5) |\n| **AMD → DEL** | 10 Aug 07:30 → 09:15 (1h 45m) | 54 EUR | [Book 
Now](https://kiwi.com/u/r5qj5c) |\n| **AMD → DEL** | 10 Aug 09:50 → 11:35 (1h 45m) | 54 EUR | [Book 
Now](https://kiwi.com/u/2fkz5j) |\n| **AMD → DEL** | 10 Aug 14:35 → 16:20 (1h 45m) | 54 EUR | [Book 
Now](https://kiwi.com/u/8n7qu4) |\n\n### Summary\n*   **Cheapest Options:** There are several direct flights priced
at **54 EUR**.\n*   **Shortest Options:** The flights departing at 19:25 and 21:50 are the shortest at **1 hour and
40 minutes**.\n\nHave a wonderful trip to Delhi! Fun fact: Delhi is home to the Humayun's Tomb, which was the first
garden-tomb on the Indian subcontinent and inspired the design of the Taj Mahal.",
        'extras': {'signature': 'EjQKMgERTTIPx4Q1eLBAkxBtFQ0r2KcvNKCu9z3ctVwT7XfdlM2F1v81U+j3CVmHLxl+ywVv'}
    }
]